# Periodic Solution Ensemble Benchmark

This notebook compares candidate solutions for the periodic branch, focusing on whether periodic baselines are subtracted accurately enough to recover one-off dimming events.

It uses the same hard synthetic population family as `periodic_branch_simulation_benchmark.ipynb`: multi-camera, seasonal, unevenly sampled, imperfectly periodic light curves with camera offsets, amplitude modulation, phase wander, slow trends, quasi-correlated residuals, and injected one-off dips.

The point of this notebook is not just whether the current branch detects events. It explicitly compares baseline solutions, period-error sensitivity, and a phase-local same-phase residual score that addresses the case where quiescent points overlap the event phase on other nights.

## Compared Solutions

The default comparison includes:

- `current_template_true_period`: current median phase-template branch with the true period.
- `current_template_1pct_period_error`: current template with selected-period scatter.
- `current_template_5pct_period_error`: selected-period stress test.
- `coarse_smooth_template`: fewer phase bins and stronger smoothing.
- `fine_template`: more phase bins and weaker smoothing.
- `leave_cycle_template`: phase template excluding points from the same cycle as the candidate point.
- `fourier_3harmonic`: harmonic-series periodic baseline.
- `ensemble_template_fourier`: median ensemble of current template, leave-cycle template, and Fourier baseline.
- `gp_masked_control`: current stochastic-branch baseline control.

Primary metrics to watch: `observable_recall`, `phase_local_observable_recall`, `control_false_positive_rate`, `off_target_detection_rate`, `median_resid_mad_outside_dip`, and `median_amp_recovery_ratio`.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/malca-matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from malca.evaluation.periodic_solution_ensemble_benchmark import (
    DEFAULT_SOLUTION_MODES,
    PeriodicSolutionBenchmarkConfig,
    load_periodic_solution_benchmark,
    plot_solution_heatmap,
    plot_solution_metrics,
    plot_solution_trial_diagnostic,
    run_periodic_solution_benchmark,
    select_example_trials,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## Configuration

Set `MALCA_PERIODIC_SOLUTION_SMOKE=1` for a quick execution check. The default notebook run is moderate, not production scale. Increase `MALCA_PERIODIC_SOLUTION_N_TRIALS` once the smoke run is clean.

In [ ]:
SMOKE = os.getenv("MALCA_PERIODIC_SOLUTION_SMOKE", "0") == "1"
N_TRIALS = int(os.getenv("MALCA_PERIODIC_SOLUTION_N_TRIALS", "144" if SMOKE else "24000"))
WORKERS = int(os.getenv("MALCA_PERIODIC_SOLUTION_WORKERS", "1" if SMOKE else "8"))
FORCE = os.getenv("MALCA_PERIODIC_SOLUTION_FORCE", "0") == "1"
RUN_TAG = os.getenv("MALCA_PERIODIC_SOLUTION_RUN_TAG", "smoke" if SMOKE else None)

modes_text = os.getenv("MALCA_PERIODIC_SOLUTION_MODES", "")
MODE_NAMES = tuple(item.strip() for item in modes_text.split(",") if item.strip()) or DEFAULT_SOLUTION_MODES

config = PeriodicSolutionBenchmarkConfig(
    n_trials=N_TRIALS,
    workers=WORKERS,
    force=FORCE,
    run_tag=RUN_TAG,
    mode_names=MODE_NAMES,
    show_progress=True,
)
config

## Run Or Load

This writes:

- `trial_design.parquet`
- `solution_results.parquet`
- `solution_results_with_design.parquet`
- `summary_tables/*.parquet`
- `summary_tables/*.csv`

In [ ]:
run = run_periodic_solution_benchmark(config)
run.run_dir

## Overall Ranking

Read this first. A useful solution should improve recall without increasing off-target detections or false positives, and should preserve injected dip amplitude after baseline subtraction.

In [ ]:
overall_cols = [
    "mode",
    "baseline_strategy",
    "n",
    "observable_recall",
    "phase_local_observable_recall",
    "precision_by_trial",
    "control_false_positive_rate",
    "off_target_detection_rate",
    "median_resid_mad_outside_dip",
    "median_amp_recovery_ratio",
    "median_phase_local_truth_peak_snr",
]

overall = run.summary_overall.loc[:, overall_cols].sort_values(
    ["observable_recall", "control_false_positive_rate", "off_target_detection_rate"],
    ascending=[False, True, True],
)
overall

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_solution_metrics(run.summary_overall, ax=ax)
fig.tight_layout()

## Baseline Quality

This table isolates the baseline question from the event trigger question. Low outside-dip residual MAD/RMS and high amplitude recovery are what you want before tuning detection thresholds.

In [ ]:
baseline_cols = [
    "mode",
    "median_baseline_mae_outside_dip",
    "median_baseline_rmse_outside_dip",
    "median_resid_rms_outside_dip",
    "median_resid_mad_outside_dip",
    "median_amp_recovery_ratio",
    "median_same_phase_truth_contrast_mag",
]
run.summary_overall.loc[:, baseline_cols].sort_values("median_resid_mad_outside_dip")

## Slice Tables

Use these to identify where a solution fails: small amplitude dips, sparse sampling, long periods, broad dips, or waveform morphology.

In [ ]:
run.summary_slices["by_mode_and_amp"].sort_values(["mode", "dip_amp_bin"]).head(60)

In [ ]:
run.summary_slices["by_mode_and_period"].sort_values(["mode", "period_bin"]).head(80)

## Recovery And Baseline Heatmaps

These show recall and amplitude preservation by dip amplitude and period. Start with the top-ranked modes from the overall table, then add more modes if needed.

In [ ]:
modes_to_plot = list(overall["mode"].head(4))
fig, axes = plt.subplots(len(modes_to_plot), 2, figsize=(15, 4.2 * len(modes_to_plot)))
axes = np.atleast_2d(axes)
for row_idx, mode in enumerate(modes_to_plot):
    plot_solution_heatmap(
        run.solution_results,
        metric="target_recovered",
        mode=mode,
        ax=axes[row_idx, 0],
        title=f"Target recovery: {mode}",
    )
    plot_solution_heatmap(
        run.solution_results,
        metric="amp_recovery_ratio",
        mode=mode,
        ax=axes[row_idx, 1],
        title=f"Amplitude recovery ratio: {mode}",
    )
fig.tight_layout()

## Failure Tables

`phase_local_only` rows are especially important: those are cases where the same-phase residual contrast is present but the current Bayesian/run path did not recover the injected dip.

In [ ]:
best_mode = str(overall.iloc[0]["mode"])
current_mode = "current_template_true_period" if "current_template_true_period" in set(run.solution_results["mode"]) else best_mode

cols = [
    "trial_id",
    "mode",
    "dip_class",
    "period_days",
    "n_points_actual",
    "dip_amp_mag",
    "dip_sigma_days",
    "truth_peak_snr_actual",
    "target_recovered",
    "phase_local_detected",
    "phase_local_truth_peak_snr",
    "same_phase_truth_contrast_mag",
    "amp_recovery_ratio",
    "baseline_source",
]

misses = run.solution_results[
    run.solution_results["mode"].eq(current_mode)
    & run.solution_results["has_dip"].fillna(False).astype(bool)
    & run.solution_results["truth_observable_actual"].fillna(False).astype(bool)
    & ~run.solution_results["target_recovered"].fillna(False).astype(bool)
].copy()

misses.loc[:, cols].sort_values("phase_local_truth_peak_snr", ascending=False).head(25)

In [ ]:
phase_local_only = misses[misses["phase_local_detected"].fillna(False).astype(bool)].copy()
phase_local_only.loc[:, cols].sort_values("phase_local_truth_peak_snr", ascending=False).head(25)

In [ ]:
false_positives = run.solution_results[
    run.solution_results["mode"].eq(best_mode)
    & run.solution_results["false_positive"].fillna(False).astype(bool)
].copy()

false_positives.loc[:, cols].sort_values("dip_bayes_factor", ascending=False).head(25)

## Example Diagnostics

Pick a mode and inspect representative successes, misses, off-target detections, false positives, and phase-local-only cases.

In [ ]:
EXAMPLE_MODE = best_mode
examples = select_example_trials(run.solution_results, EXAMPLE_MODE)
examples

In [ ]:
for label, trial_id in examples.items():
    if trial_id is None:
        continue
    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)
    plot_solution_trial_diagnostic(run, int(trial_id), mode=EXAMPLE_MODE, ax=axes)
    fig.suptitle(f"{label}: trial {trial_id} | {EXAMPLE_MODE}", y=1.0)
    fig.tight_layout()

## Interpretation Checklist

- If `current_template_true_period` has weak recall, the current periodic branch is baseline or scorer limited even with perfect period selection.
- If true-period recall is good but 1%/5% period-error modes collapse, period selection quality is the bottleneck.
- If `leave_cycle_template` or `ensemble_template_fourier` improves `median_amp_recovery_ratio`, the current template is absorbing part of one-off dips.
- If `phase_local_observable_recall` is much higher than `observable_recall`, the baseline contains the event information but the stochastic event scorer/run filters are suppressing it.
- If false positives rise in Fourier or fine-template modes, the model is likely leaving structured phase residuals or overfitting sparse phase regions.
- If GP control wins on periodic curves, the folding assumptions are too brittle for the simulated cadence/period regime and the pregate needs stricter period-quality requirements.